# Linear Transformer for Blood Pressure Estimation

This notebook implements a Linear Transformer model for blood pressure estimation from pulse waveforms, with attention masks initialized from a pretrained LodeSTAR model.

In [65]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt
from tqdm import tqdm
import polars as pl
from functools import lru_cache
# import wandb  # Weights & Biases for experiment tracking
from sklearn.metrics import mean_squared_error
from LinearTransformerModel import LinearTransformer

%load_ext autoreload
%autoreload 2   


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

In [76]:
class Config:
    # Data parameters
    seq_len =20
    batch_size = 8
    num_workers = 4
    
    # Model parameters
    input_dim = 2  # Assuming input has 2 channels (pulse and motion)
    embed_dim = 128
    num_heads = 8
    num_layers = 6
    hidden_dim = 512
    dropout = 0.1
    
    # Training parameters
    lr = 1e-4
    weight_decay = 1e-5
    epochs = 100
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Paths
    lodestar_weights_path = 'path_to_lodestar_weights.pth'
    checkpoint_dir = 'checkpoints'
    
    # Weights for loss components
    bp_loss_weight = 1.0
    pulse_loss_weight = 0.5
    
    # Initialize Weights & Biases
    use_wandb = True
    project_name = "bp_estimation_transformer"
    
config = Config()

# Create checkpoint directory
os.makedirs(config.checkpoint_dir, exist_ok=True)

'''Read controids trajectories from path to a numpy array

Input : str path to load

Output : np.array Dimension: [segmentLength, numCentroids, 2]  
'''
def read_from_csv(centroids_traj_fname):

    cents_pos = pd.read_csv(centroids_traj_fname)

    if 'Unnamed: 0' in cents_pos.columns:
        centroids = cents_pos.drop('Unnamed: 0', axis=1)
    else: 
        centroids  = cents_pos
    centroidSnaps     = centroids.values
    odd_columns       = centroidSnaps[:, 1::2]
    even_columns      = centroidSnaps[:, ::2]
    if odd_columns.shape != even_columns.shape:
        min_cols = min(odd_columns.shape[1], even_columns.shape[1])
        odd_columns  = odd_columns[:, :min_cols]
        even_columns = even_columns[:, :min_cols]
    centroid_trajectories = np.stack([odd_columns, even_columns], axis=2)   
    return centroid_trajectories

In [77]:
DataSegments = pd.read_csv(os.path.expanduser('~/DynoNAS/DatabaseTables/DataSegments.csv'))
display(DataSegments.head())

sid = '2024y_03m_14d_12h_14m_19s_787ms_358us_tracking_0'
            
cond1 = DataSegments['unique_file_id'] == sid.partition('us')[0] + sid.partition('us')[1]
cond2  = DataSegments['relative_idx'] == int(sid[-1])
cond3 = DataSegments['type'] == 'tracking'
subject_id = DataSegments[cond1 & cond2 & cond3]
display(subject_id)

,id,subject_id,record_id,nas_path,unique_file_id,visit_id,visit_type,hospital_name,study_name,ward_name,ground_truth_type,type,start_idx,end_idx,relative_idx,schema_version
0,1,1603,6,DynoNAS/Clinical_Studies/LSU/PID1603,2023y_09m_12d_14h_37m_38s_735ms_830us,2,in_patient,LSU,NaN,NaN,NaN,tracking,15051,20863,0,1
1,2,1603,6,DynoNAS/Clinical_Studies/LSU/PID1603,2023y_09m_12d_14h_37m_38s_735ms_830us,2,in_patient,LSU,NaN,NaN,NaN,upsweep,7238,8972,0,1
2,3,1603,6,DynoNAS/Clinical_Studies/LSU/PID1603,2023y_09m_12d_14h_37m_38s_735ms_830us,2,in_patient,LSU,NaN,NaN,NaN,downsweep,8975,10710,0,1
3,4,1602,7,DynoNAS/Clinical_Studies/LSU/PID1602,2023y_09m_12d_21h_25m_30s_138ms_525us,3,in_patient,LSU,NaN,NaN,NaN,tracking,52929,70437,0,1
4,5,1602,7,DynoNAS/Clinical_Studies/LSU/PID1602,2023y_09m_12d_21h_25m_30s_138ms_525us,3,in_patient,LSU,NaN,NaN,NaN,tracking,76698,88094,1,1


,id,subject_id,record_id,nas_path,unique_file_id,visit_id,visit_type,hospital_name,study_name,ward_name,ground_truth_type,type,start_idx,end_idx,relative_idx,schema_version
626,627,6,146,DynoNAS/Clinical_Studies/ClevelandClinic/CCF_D...,2024y_03m_14d_12h_14m_19s_787ms_358us,60,in_patient,ClevelandClinic,NaN,NaN,IAP,tracking,23555,36817,0,3


## Dataset and DataLoader

In [78]:



class BPDataset(Dataset):
    def __init__(
        self,
        points_data_path,
        BP_data_path,
        DataSegments,              # Polars DataFrame recommended
        window_size=120,
        overlap=20,
        use_overlap=True,
        transform=None,
    ):
        self.centroids_dir = points_data_path
        self.bp_dir = BP_data_path
        self.window_size = int(window_size)
        self.overlap = int(overlap)
        self.use_overlap = use_overlap
        self.transform = transform
        self.stride = self.window_size - self.overlap if use_overlap else self.window_size

        # ------------------------------------------------------------------
        # Identify subjects
        # ------------------------------------------------------------------
        files = os.listdir(self.centroids_dir)
        self.subject_ids = sorted(
            f.split("_r0.05_CentroidPositionsLowPass.csv")[0]
            for f in files
            if f.endswith("_r0.05_CentroidPositionsLowPass.csv")
        )

        # ------------------------------------------------------------------
        # Build window index (lightweight metadata only)
        # ------------------------------------------------------------------
        self.index = []

        for sid in self.subject_ids:
            try:
                clean_path = os.path.join(
                    self.centroids_dir, f"{sid}_r0.05_CentroidPositionsLowPass.csv"
                )
                easy_path = os.path.join(
                    self.centroids_dir, f"{sid}_r0.35_CentroidPositionsLowPass.csv"
                )
                hard_path = os.path.join(
                    self.centroids_dir, f"{sid}_r0.65_CentroidPositionsLowPass.csv"
                )
                noise_path = os.path.join(
                    self.centroids_dir, f"{sid}_r0.95_CentroidPositionsLowPass.csv"
                )

                subject_id = (
                    DataSegments
                    .filter(
                        (pl.col("unique_file_id") == (sid.partition("us")[0] + "us"))
                        & (pl.col("relative_idx") == int(sid[-1]))
                        & (pl.col("type") == "tracking")
                    )
                    .select("subject_id")
                    .item()
                )

                bp_path = os.path.join(
                    self.bp_dir, f"{subject_id}_{sid}", f"{sid}_Time.csv"
                )

                # Determine minimum synchronized length without loading full arrays
                min_len = min(
                    pl.scan_csv(clean_path).select(pl.len()).collect().item(),
                    pl.scan_csv(easy_path).select(pl.len()).collect().item(),
                    pl.scan_csv(hard_path).select(pl.len()).collect().item(),
                    pl.scan_csv(noise_path).select(pl.len()).collect().item(),
                    pl.scan_csv(bp_path).select(pl.len()).collect().item(),
                )

                for start in range(0, min_len - self.window_size + 1, self.stride):
                    self.index.append(
                        (sid, start)
                    )

            except Exception as e:
                print(f"Skipping subject {sid}: {e}")

    # ----------------------------------------------------------------------
    # Cached loaders (per worker)
    # ----------------------------------------------------------------------
    @lru_cache(maxsize=32)
    def _load_centroid_csv(self, path):
        return pl.read_csv(path).to_numpy()

    @lru_cache(maxsize=32)
    def _load_bp_csv(self, path):
        return (
            pl.read_csv(path)
            .select("PULSE_Y")
            .to_numpy()
            .squeeze()
        )

    # ----------------------------------------------------------------------
    def __len__(self):
        return len(self.index)

    # ----------------------------------------------------------------------
    def __getitem__(self, idx):
        sid, start = self.index[idx]
        end = start + self.window_size

        clean = self._load_centroid_csv(
            os.path.join(self.centroids_dir, f"{sid}_r0.05_CentroidPositionsLowPass.csv")
        )[start:end]

        easy = self._load_centroid_csv(
            os.path.join(self.centroids_dir, f"{sid}_r0.35_CentroidPositionsLowPass.csv")
        )[start:end]

        hard = self._load_centroid_csv(
            os.path.join(self.centroids_dir, f"{sid}_r0.65_CentroidPositionsLowPass.csv")
        )[start:end]

        noise = self._load_centroid_csv(
            os.path.join(self.centroids_dir, f"{sid}_r0.95_CentroidPositionsLowPass.csv")
        )[start:end]

        bp = self._load_bp_csv(
            os.path.join(self.bp_dir, f"{sid}", f"{sid}_Time.csv")
        )[start:end]

        if self.transform:
            clean = self.transform(clean)
            easy = self.transform(easy)
            hard = self.transform(hard)
            noise = self.transform(noise)

        return (
            torch.from_numpy(clean).float(),
            torch.from_numpy(easy).float(),
            torch.from_numpy(hard).float(),
            torch.from_numpy(noise).float(),
            torch.from_numpy(bp).float(),
        )


In [ ]:
   
# Create datasets and dataloaders
full_dataset = BPDataset(points_data_path= os.path.expanduser('~/DynoNAS/Peter/Research_data/Synthetic_Videos_Noise_Mixture_Protocol'),
                        BP_data_path= os.path.expanduser('~/DynoNAS/ProcessedData/BeatDetector_v3'),
                        DataSegments=DataSegments,
                        window_size=config.seq_len,
                        overlap=5,
                        use_overlap=True,
                        transform=None)  
print("Dataset length:", len(full_dataset))

num_cpus = os.cpu_count() or 1

# 2. Split the dataset (0.7 Train, 0.3 Test)
train_size = int(0.7 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

# 3. Create DataLoaders
train_loader = DataLoader(train_dataset,
                          batch_size=8,
                          shuffle=True,
                          num_workers=min(4, num_cpus), # Conservative start
                          pin_memory=True,              # Faster data transfer to GPU
                          persistent_workers=True       # Keeps workers alive between epochs for speed)
                        )

test_loader = DataLoader(test_dataset,
                         batch_size=8,
                         shuffle=False,
                         num_workers=min(4, num_cpus), # Conservative start
                         pin_memory=True,              # Faster data transfer to GPU
                         persistent_workers=True       # Keeps workers alive between epochs for speed)
                        )

Skipping subject 2024y_03m_14d_12h_14m_19s_787ms_358us_tracking_0: Index(...) must be called with a collection of some kind, <Expr ['[([([(col("unique_file_id")) =…'] at 0x7E661B182AD0> was passed
Skipping subject 2024y_03m_14d_12h_37m_39s_532ms_546us_tracking_0: Index(...) must be called with a collection of some kind, <Expr ['[([([(col("unique_file_id")) =…'] at 0x7E661B183340> was passed
Skipping subject 2024y_03m_14d_12h_37m_39s_532ms_546us_tracking_1: Index(...) must be called with a collection of some kind, <Expr ['[([([(col("unique_file_id")) =…'] at 0x7E661B182B30> was passed
Skipping subject 2024y_03m_14d_13h_51m_42s_069ms_802us_tracking_0: Index(...) must be called with a collection of some kind, <Expr ['[([([(col("unique_file_id")) =…'] at 0x7E661B182AA0> was passed
Skipping subject 2024y_03m_14d_13h_51m_42s_069ms_802us_tracking_1: Index(...) must be called with a collection of some kind, <Expr ['[([([(col("unique_file_id")) =…'] at 0x7E661B1836D0> was passed
Skipping subjec

ValueError: num_samples should be a positive integer value, but got num_samples=0

## Model Initialization

In [24]:
def initialize_model():
    """Initialize the model and load pretrained attention masks if available"""
    model = LinearTransformer(
        input_dim=config.input_dim,
        embed_dim=config.embed_dim,
        num_heads=config.num_heads,
        num_layers=config.num_layers,
        hidden_dim=config.hidden_dim,
        dropout=config.dropout,
        max_seq_len=config.seq_len
    )
    
    # Load pretrained attention masks from LodeSTAR if available
    if os.path.exists(config.lodestar_weights_path):
        try:
            model.load_pretrained_attention_masks(config.lodestar_weights_path)
            print("Successfully loaded attention masks from LodeSTAR model")
        except Exception as e:
            print(f"Error loading attention masks: {e}")
    
    return model.to(config.device)

model = initialize_model()

## Training Setup

In [26]:
def setup_training(model):
    """Set up optimizer, scheduler, and loss functions"""
    # Optimizer
    optimizer = optim.AdamW(
        model.parameters(),
        lr=config.lr,
        weight_decay=config.weight_decay
    )
    
    # Learning rate scheduler
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=5,
    )
    
    # Loss functions
    bp_criterion = nn.MSELoss()  # For BP estimation
    pulse_criterion = nn.MSELoss()  # For pulse waveform prediction
    
    return optimizer, scheduler, bp_criterion, pulse_criterion

optimizer, scheduler, bp_criterion, pulse_criterion = setup_training(model)

## Training Loop

In [27]:
def train_epoch(model, dataloader, optimizer, bp_criterion, pulse_criterion, epoch):
    model.train()
    total_loss = 0.0
    bp_loss_total = 0.0
    pulse_loss_total = 0.0
    
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1} [Train]")
    
    for batch in progress_bar:
        # Move data to device
        inputs = batch['input'].to(config.device)
        bp_targets = batch['bp'].to(config.device)
        pulse_targets = batch['pulse'].to(config.device)
        
        # Forward pass
        optimizer.zero_grad()
        bp_pred, pulse_pred = model(inputs)
        
        # Calculate losses
        bp_loss = bp_criterion(bp_pred, bp_targets)
        pulse_loss = pulse_criterion(pulse_pred, pulse_targets)
        
        # Combine losses
        loss = config.bp_loss_weight * bp_loss + config.pulse_loss_weight * pulse_loss
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        # Update metrics
        total_loss += loss.item()
        bp_loss_total += bp_loss.item()
        pulse_loss_total += pulse_loss.item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': total_loss / (progress_bar.n + 1),
            'bp_loss': bp_loss_total / (progress_bar.n + 1),
            'pulse_loss': pulse_loss_total / (progress_bar.n + 1)
        })
    
    # Calculate average losses
    avg_loss = total_loss / len(dataloader)
    avg_bp_loss = bp_loss_total / len(dataloader)
    avg_pulse_loss = pulse_loss_total / len(dataloader)
    
    # Log average losses
    print(f"Epoch {epoch+1} [Train] - Avg Loss: {avg_loss:.4f}, Avg BP Loss: {avg_bp_loss:.4f}, Avg Pulse Loss: {avg_pulse_loss:.4f}")
    
    return avg_loss, avg_bp_loss, avg_pulse_loss
